# 🔎 Zacznij tutaj
# 🔎 05. Supervised Semantic Differential (SSD)
**Metody mieszane w analizie tekstu: od słowników do BERT**

SSD jest ostatnią i najbardziej zaawansowaną metodą warsztatu. Pytamy, czy sposób
przedstawiania **jednego pojęcia** w otaczającym języku wiąże się z inną zmienną.
BERTopic szuka powtarzalnych struktur w korpusie. Słownik liczy ustalone kategorie,
a sentyment szacuje ton. Sam embedding daje reprezentację, lecz nie wyznacza
kierunku związanego z wynikiem osobowości.

Otwórz ten plik w [Google Colab](https://colab.research.google.com/), wybierz
**Plik → Prześlij notatnik**, a następnie uruchamiaj komórki od góry.
W sekcji 2 dane pobiorą się automatycznie z repozytorium prowadzącego. To bezstratna konwersja arkusza
`final2.xlsx` (Sheet1) dostarczonego z zadaniem. Nie potrzebujesz lokalnego Pythona.
Pierwsze uruchomienie pobiera biblioteki i model. CPU wystarcza.

**Ograniczenie danych:** opis wskazanego zbioru Kaggle mówi o wynikach obliczonych
z ocen odpowiedzi tekstowych. Nie są udokumentowanym, niezależnym pomiarem Big Five.
Nie znamy dokładnego algorytmu oceniania, sędziów, rekrutacji ani udziału generowania
syntetycznego. Analiza jest demonstracją związku tekst–etykieta, nie walidacją osobowości.


**Metoda:** używamy autorskiego pakietu `ssdiff==3.0.0`, procedury **PCA + OLS**.
PCV powstaje ze statycznych wektorów słów kontekstu (GloVe), z wagami SIF i pominięciem
samego trafienia leksykonu. Interpretujemy słowa przy ±β, ich klastry i rzeczywiste teksty.
Na potrzeby zajęć wybieramy mały model GloVe 50d. Liczbę składowych wybiera autorski PCA sweep w ustalonym zakresie K=1,3,…,15.
To jawne ustawienia demonstracji, nie parametry konkretnego badania Plisieckiego.
Wersja v3 publikacji uwzględnia PCA sweep. Pakiet oferuje też odrębny wariant PLS.
W tym module uczymy oryginalnej ścieżki PCA + OLS.


# 🧭 Jak wykonać ten notebook

1. Otwórz plik `.ipynb` w Google Colab i zapisz własną kopię na Dysku Google.
2. **Wystarczy CPU.** Komórka to jeden blok tekstu albo kodu. Kod uruchamiasz przyciskiem ▶ po lewej lub Shift+Enter.
3. Uruchom pierwszą komórkę kodu. Jeżeli zainstaluje biblioteki i pokaże 🔄, uruchom ponownie sesję z menu **Środowisko wykonawcze**. Potem zacznij od pierwszej komórki. Restart zachowuje pliki, ale usuwa zmienne z pamięci.
4. Wykonuj kod **od góry, bez pomijania komórek**. Dane pobiorą się automatycznie z [repozytorium prowadzącego](https://github.com/bartlomiejnowak-ux/PSPS-2026). Domyślnie niczego nie wgrywasz. W razie awarii pobierania komórka podaje instrukcję ręcznego wgrania.
5. Poczekaj, aż obracający się znacznik przy komórce zniknie. Pierwsze pobranie modelu i obliczenia mogą potrwać kilka minut, a BERTopic dłużej. Nie klikaj wielokrotnie ▶.
6. ✅ oznacza sukces, 🔎 wskazuje co przeczytać, ⚠️ ważne ograniczenie, ✏️ ćwiczenie. Emoji nie zmieniają działania kodu.
7. Na końcu pobierz ZIP wyników. Sam zapis notebooka na Dysku nie zachowuje plików z tymczasowej sesji.


### 🛠️ Gdy coś nie działa

| Objaw | Co zrobić |
|---|---|
| `NameError` lub „nie zdefiniowano” | Pominięto wcześniejszy krok albo zrestartowano sesję. Wykonaj kod od początku. |
| `ModuleNotFoundError` / błąd wersji biblioteki | Uruchom instalację, zrestartuj sesję i wykonaj komórki od góry. |
| Brak pliku / zła kolumna | Ponów komórkę pobierania danych. Awaryjnie wybierz `final2.xlsx` z repozytorium. |
| Błąd pobierania modelu | Sprawdź połączenie, zaczekaj i ponów komórkę pobierania. Nie zmieniaj nazwy modelu. |
| Błąd po zmianie parametru | Cofnij zmianę lub przywróć wartości pokazane w komentarzach i wykonaj dalsze komórki kolejno. |
| Sesja wygasła | Połącz ponownie, uruchom notebook od początku i ponownie wykonaj komórkę pobierania danych. |

Nie przechodź dalej po czerwonym błędzie. Czytaj ostatnią linijkę komunikatu. W tej kopii wyniki pojawią się dopiero po uruchomieniu kodu. Liczby na slajdach pochodzą ze sprawdzonego wcześniejszego wykonania; przy innych ustawieniach lub środowisku wynik może się różnić.

## 📖 Słowa i skróty używane w tym module

- **SSD:** metoda badania związku kontekstu jednego pojęcia z inną zmienną.
- **Y:** wynik, który model przewiduje; tutaj ocena sumienności.
- **PCV:** wektor pojęcia w jednym rekordzie, obliczony z jego kontekstów.
- **GloVe:** gotowe, statyczne wektory słów: dane słowo ma jeden wektor.
- **SIF:** ważenie kontekstu tak, aby bardzo częste słowa miały mniejszą wagę.
- **L2:** przeskalowanie wektora do długości 1.
- **ABTT:** usunięcie dominującego wspólnego kierunku z wektorów.
- **PCA / K:** redukcja liczby wymiarów / liczba zachowanych składowych.
- **PCA sweep:** porównanie kilku wartości K według autorskich kryteriów.
- **OLS:** regresja minimalizująca sumę kwadratów błędów przewidywania.
- **β / gradient / backprojekcja:** kierunek wzrostu przewidywanego Y / ten sam kierunek / powrót współczynników do przestrzeni słów.
- **CV / GroupKFold:** sprawdzenie predykcji na odłożonych danych / podział chroniący grupy przed rozdzieleniem między trening i test.
- **R²:** porównanie błędu modelu z odchyleniami wyników od średniej ocenianego zbioru; 1 oznacza brak błędu, 0 brak poprawy, wynik ujemny gorszy wynik. W walidacji średnia testu służy tylko do obliczenia tej miary. Osobny model odniesienia (baseline) przewiduje średnią treningu.
- **MAE:** średni bezwzględny błąd w punktach skali Y; mniej znaczy lepiej.
- **Baseline:** prosty punkt odniesienia: średnia z części treningowej.
- **Cosine / alignment:** podobieństwo kierunków / zgodność wektora rekordu z kierunkiem β.
- **Silhouette:** miara zwartości i oddzielenia klastrów; sama nie potwierdza sensu psychologicznego.
- **Big Five:** otwartość, sumienność, ekstrawersja, ugodowość i neurotyczność.

## 🔎 1. Instalacja bibliotek

**▶️ Co teraz robimy?** Instalujemy biblioteki w sesji Colab.

**🎯 Po co to robimy?** Dalsze komórki używają tego samego zestawu narzędzi.

### ▶️ Krok kodu 1

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** ✅ Biblioteki gotowe albo instrukcja jednorazowego restartu 🔄.

In [ ]:
import sys, subprocess, importlib.util, importlib.metadata as metadata
from pathlib import Path
IN_COLAB = importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
PACKAGES = ['openpyxl==3.1.5', 'ssdiff==3.0.0', 'pandas==3.0.5', 'numpy==2.5.3', 'matplotlib==3.11.2', 'scipy==1.18.1', 'scikit-learn==1.9.1', 'spacy==3.8.16', 'https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl']
if sys.version_info < (3, 12):
    raise RuntimeError('⛔ Ten zestaw wersji wymaga Python 3.12 lub nowszego. Wybierz zgodne środowisko.')
def installed(spec):
    name, expected = ('en-core-web-sm', '3.8.0') if spec.startswith('https:') else spec.split('==')
    try: return metadata.version(name) == expected
    except metadata.PackageNotFoundError: return False
needed = [spec for spec in PACKAGES if not installed(spec)]
if needed and IN_COLAB:
    print('⏳ Instalacja bibliotek. Poczekaj na zakończenie tej komórki.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES])
    raise RuntimeError('🔄 Instalacja zakończona. Uruchom ponownie sesję z menu Środowisko wykonawcze, a następnie wykonaj komórki od początku. To jednorazowy krok po instalacji.')
if needed:
    raise RuntimeError('⛔ Brakuje wymaganych wersji. Lokalnie użyj pliku requirements właściwego modułu. W Colab komórka instaluje je sama. Braki: ' + ', '.join(needed))
print('✅ Biblioteki gotowe. Możesz uruchomić następną komórkę.')


### ▶️ Krok kodu 2

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** przygotowanie funkcji lub ustawień do dalszych kroków; brak wydruku jest prawidłowy.

In [ ]:
from pathlib import Path
from collections import Counter
import re, json, hashlib, platform, importlib.metadata as meta, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_absolute_error
from ssdiff import Embeddings, Corpus, SSD
from ssdiff.utils.vectors import build_doc_vectors, compute_global_sif
OUT = Path('05_SSD_results')
OUT.mkdir(exist_ok=True)
plt.rcParams.update({'figure.figsize': (9,4), 'font.size': 12})

**✅ Co powinno wyjść?** Komunikat Setup ready.

**🔎 Jak to interpretować?** Biblioteki zamieniają tekst w wektory i dopasowują prosty model.

**⚠️ Na co uważać?** Po zmianie zainstalowanych wersji Colab może poprosić o restart sesji. Wtedy uruchom komórki ponownie. Lokalny test nie odtwarza infrastruktury Google.

## 🔎 2. Pobranie danych

**▶️ Co teraz robimy?** Pobieramy Excel z GitHuba i zamieniamy go na tabelę CSV.

**🎯 Po co to robimy?** Każdy wiersz pozostaje powiązany ze swoim ID i wynikami.

### ▶️ Krok kodu 3

Uruchom raz i poczekaj. **Oczekiwany efekt:** automatyczne pobranie danych i komunikat ✅. Przy awarii ustaw w kodzie `DATA_SOURCE = 'upload'` i wybierz **`final2.xlsx`** z repozytorium prowadzącego.

In [ ]:
# 📥 Dane z repozytorium prowadzącego. Domyślnie niczego nie wgrywasz ręcznie.
import io, urllib.request, hashlib
DATA_SOURCE = 'github'  # awaryjnie zmień na 'upload' i uruchom komórkę ponownie
GITHUB_COMMIT = '1cb710169713a4f8ea5a0d839be6d6e1df8ab078'
GITHUB_URL = f'https://raw.githubusercontent.com/bartlomiejnowak-ux/PSPS-2026/{GITHUB_COMMIT}/final2.xlsx'
EXPECTED_SHA256 = '49bb427bccda8648f1e11f5440e2085c4b23d82099f66b7400f2d5cc05acf0ed'
INPUT_NAME = 'final2.csv'
if IN_COLAB:
    if DATA_SOURCE == 'github':
        print('⏳ Pobieranie danych z GitHub…')
        try:
            with urllib.request.urlopen(GITHUB_URL, timeout=60) as response:
                data_bytes = response.read()
        except Exception as exc:
            raise RuntimeError('⛔ Nie udało się pobrać danych. Sprawdź internet i ponów tę komórkę. Awaryjnie ustaw DATA_SOURCE = "upload" i wybierz plik z repozytorium prowadzącego.') from exc
        if hashlib.sha256(data_bytes).hexdigest() != EXPECTED_SHA256:
            raise ValueError('⛔ Pobrany plik nie zgadza się ze sprawdzoną wersją. Nie kontynuuj analizy na tym pliku.')
    elif DATA_SOURCE == 'upload':
        from google.colab import files
        print('📂 Wybierz final2.xlsx z repozytorium prowadzącego.')
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError('⛔ Wybierz dokładnie jeden plik i ponów tę komórkę.')
        data_bytes = next(iter(uploaded.values()))
    else:
        raise ValueError('⛔ DATA_SOURCE musi mieć wartość "github" albo "upload".')
    try:
        source_df = pd.read_excel(io.BytesIO(data_bytes), sheet_name='Sheet1')
    except Exception as exc:
        raise ValueError('⛔ Nie można odczytać pliku. Wgraj final2.xlsx z repozytorium prowadzącego.') from exc
    missing_columns = {'Agreeableness_Score', 'q2_response', 'q4_response', 'q1_response', 'Neuroticism_Score', 'q3_response', 'ID', 'Extraversion_Score', 'Conscientiousness_Score', 'Openness_Score', 'q5_response'} - set(source_df.columns)
    if source_df.empty or missing_columns:
        raise ValueError(f'⛔ Pusty lub niewłaściwy plik. Brakujące kolumny: {sorted(missing_columns)}')
    data_folder = Path('data')
    data_folder.mkdir(parents=True, exist_ok=True)
    source_df.to_csv(data_folder / INPUT_NAME, index=False)
    print(f'✅ Dane gotowe: {len(source_df)} wierszy. Źródło: {DATA_SOURCE}.')

DATA_PATH = Path('data/final2.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('⛔ Brak danych. Ponów pobieranie, a lokalnie sprawdź data/final2.csv.')
df = pd.read_csv(DATA_PATH)
display(df.head(3))


**✅ Co powinno wyjść?** 299 wierszy i 14 kolumn dla dołączonego final2.csv.

**🔎 Jak to interpretować?** ID jest identyfikatorem rekordu w pliku. Nie potwierdza tożsamości ani niezależności realnych uczestników.

**⚠️ Na co uważać?** Używamy wskazanego zbioru Big Five, a nie korpusu zdrowia psychicznego z poprzednich modułów. Nie wymagamy konta Kaggle.

## 🔎 3. Kontrola danych

**▶️ Co teraz robimy?** Sprawdzamy kolumny, braki, skalę i powtórzenia.

**🎯 Po co to robimy?** Model ma sens tylko przy znanej jednostce obserwacji i kodowaniu Y.

### ▶️ Krok kodu 4

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
TEXT_COLUMNS = [f'q{i}_response' for i in range(1, 6)]
TRAITS = ['Openness_Score', 'Conscientiousness_Score', 'Extraversion_Score',
          'Agreeableness_Score', 'Neuroticism_Score']
def require_columns(frame, columns):
    missing = [c for c in columns if c not in frame.columns]
    if missing:
        print('Nie znaleziono oczekiwanej kolumny:', missing)
        print('Dostępne kolumny to:', list(frame.columns))
        raise SystemExit('Popraw nazwy kolumn lub wgraj właściwy CSV i uruchom ponownie.')
require_columns(df, ['ID', *TEXT_COLUMNS, *TRAITS])
if df.ID.isna().any() or df.ID.duplicated().any():
    raise SystemExit('ID powinno być niepuste i unikalne. Ustal jednostkę analizy przed dalszą pracą.')
audit = pd.DataFrame({'column': df.columns, 'dtype': df.dtypes.astype(str),
                      'missing': df.isna().sum(), 'unique': df.nunique()}).reset_index(drop=True)
display(audit)
audit.to_csv(OUT/'dataset_audit.csv', index=False)

### ▶️ Krok kodu 5

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
# Odczyt rzeczywistego kodowania, bez zakładania ciągłej skali.
trait_distribution = pd.concat([
    df[c].value_counts(dropna=False).sort_index().rename('n').rename_axis('score')
      .reset_index().assign(trait=c) for c in TRAITS], ignore_index=True)
display(trait_distribution)
trait_distribution.to_csv(OUT/'trait_distributions.csv', index=False)
print('Powtórzone całe wiersze:', int(df.duplicated().sum()))
print('Liczba różnych tekstów:', df[TEXT_COLUMNS].nunique().to_dict())

Pytania w opisie Kaggle odnoszą się kolejno do uczenia się/nowości (q1), zadań i terminów (q2),
działania w grupie (q3), konfliktów (q4) i stresu/emocji (q5). Treści są krótkie, część
odpowiedzi się powtarza. To obserwacja pliku, a nie dowód określonego sposobu generowania.

Źródło danych: [Kaggle, Hassan, Personality Trait & Categories Based on Big Five](https://www.kaggle.com/datasets/hassan1212/personality-trait-and-categories-based-on-big-five).
Opis mówi o ocenach odpowiedzi w skali 1–5 i obliczanych na tej podstawie wynikach.
Nie podaje nam wystarczającej procedury, aby odtworzyć mechanizm punktacji lub potwierdzić
niezależny kwestionariusz. Lokalny plik ma zgodny schemat, ale nie zweryfikowano tożsamości
bajtowej z aktualną wersją pobrania Kaggle. Zachowujemy dostarczone dane bez zmian.

**✅ Co powinno wyjść?** Brak braków w 14 kolumnach. Skale mają wartości całkowite 1–5, Neuroticism 2–5.

**🔎 Jak to interpretować?** To zmienne porządkowe, a nie udokumentowane ciągłe wyniki testu. Regresja OLS potraktuje odstępy jako równe, jawnie jako uproszczenie.

**⚠️ Na co uważać?** Zależność wyniku od ocen tekstu stwarza kolistość. Nawet dobra walidacja predykcyjna nie dowiedzie trafności psychometrycznej.

## 🔎 4. Wybór pojęcia

**▶️ Co teraz robimy?** Sprawdzamy częstości po lematyzacji autora.

**🎯 Po co to robimy?** Pojęcie i Y wybieramy przed dopasowaniem regresji.

### ▶️ Krok kodu 6

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** przygotowanie funkcji lub ustawień do dalszych kroków; brak wydruku jest prawidłowy.

In [ ]:
# Zmień te ustawienia w ćwiczeniu. Następnie uruchom sekcje 4–14 ponownie.
TARGET_CONCEPT = 'stress'             # lemma, małe litery
BIG_FIVE_TRAIT = 'Conscientiousness_Score'
TEXT_COLUMN = 'q5_response'
CONTEXT_WINDOW = 3                   # tokeny PO preprocessingu, z każdej strony
ALTERNATIVE_WINDOW = 5
MODEL_NAME = 'glove-wiki-gigaword-50'  # statyczne wektory słów dla oryginalnego SSD
PCA_K = None                         # None = autorski PCA sweep
PCA_MAX = 15                         # zakres ustalony przed wynikami, K=1,3,...,15
SIF_A = 0.001
require_columns(df, [TEXT_COLUMN, BIG_FIVE_TRAIT])
if BIG_FIVE_TRAIT not in TRAITS:
    raise SystemExit('Wybierz wynik Big Five z listy TRAITS.')
# Autorski preprocessing: spaCy, lemmy i domyślna lista stopwords.
corpora = {c: Corpus(df[c].fillna('').astype(str).tolist(), lang='en',
                     model='en_core_web_sm', auto_download=False) for c in TEXT_COLUMNS}
corpus = corpora[TEXT_COLUMN]

### ▶️ Krok kodu 7

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
candidates = ['stress','family','relationship','conflict','self','work',
              'people','task','team','learn', TARGET_CONCEPT]
frequency = pd.DataFrame([
    {'text_column': c, 'concept': concept,
     'n_records': sum(concept in doc for doc in corp.docs),
     'n_occurrences': sum(doc.count(concept) for doc in corp.docs)}
    for c,corp in corpora.items() for concept in sorted(set(candidates))])
frequency.to_csv(OUT/'concept_frequency.csv', index=False)
display(frequency.pivot(index='concept', columns='text_column', values='n_records'))
print('Przykład preprocessingu:', corpus.docs[0])

**✅ Co powinno wyjść?** Tabela liczebności po lematyzacji i przykład tokenów wejściowych.

**🔎 Jak to interpretować?** Stress w q5 daje wspólny temat. Y=Conscientiousness dotyczy organizowania działania i pochodzi z innego pytania niż stres. To wybór koncepcyjny, nie wybór najmniejszego p.

**⚠️ Na co uważać?** Lematyzacja może połączyć stress i stressed. Liczebność może różnić się od surowych trafień całego słowa (111). Stopwords mogą zawierać negacje, więc zawsze czytamy oryginał.

## 🔎 5. Wyodrębnienie kontekstów

**▶️ Co teraz robimy?** Pokazujemy dokładne tokeny otaczające pojęcie.

**🎯 Po co to robimy?** W oryginalnym SSD samo trafienie leksykonu nie wchodzi do średniej kontekstu.

### ▶️ Krok kodu 8

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
def list_contexts(corp, frame, window):
    if not isinstance(window,int) or window < 1:
        raise SystemExit('CONTEXT_WINDOW musi być dodatnią liczbą całkowitą.')
    rows = []
    for row_idx, tokens in enumerate(corp.docs):
        for pos, token in enumerate(tokens):
            if token == TARGET_CONCEPT:
                lo, hi = max(0,pos-window), min(len(tokens),pos+window+1)
                used = [tokens[j] for j in range(lo,hi) if j != pos]
                rows.append({'row_index': row_idx, 'ID': frame.iloc[row_idx]['ID'],
                             'target_position': pos, 'tokens': used,
                             'context_tokens': ' '.join(used),
                             'full_text': frame.iloc[row_idx][TEXT_COLUMN]})
    return pd.DataFrame(rows)
contexts = list_contexts(corpus, df, CONTEXT_WINDOW)
if contexts.empty or contexts.ID.nunique() < 20:
    raise SystemExit('Za mało trafień. Wybierz częstsze pojęcie/pytanie w sekcji 4.')
print(f'✓ Target concept found in {contexts.ID.nunique()} rekordach')
display(contexts[['ID','context_tokens','full_text']].head(8))

**✅ Co powinno wyjść?** Lemmy używane do obliczeń oraz pełne, niezmienione odpowiedzi.

**🔎 Jak to interpretować?** Okno ±3 liczymy po preprocessingu, nie na oryginalnym ciągu słów. Obejmuje do sześciu sąsiadów. Kilka trafień daje kilka kontekstów.

**⚠️ Na co uważać?** Przykład lemma-context nie jest dosłownym cytatem. Pełny tekst zachowuje negacje i składnię utracone w reprezentacji. Nie łączymy odpowiedzi różnych rekordów.

## 🔎 6. Wektory słów

**▶️ Co teraz robimy?** Ładujemy statyczne embeddingi słów i normalizujemy przestrzeń.

**🎯 Po co to robimy?** Te same wektory służą budowie PCV oraz interpretacji słów przy biegunach.

### ▶️ Krok kodu 9

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** komunikat lub wyniki tekstowe pod komórką.

In [ ]:
# Pierwsze uruchomienie Colab pobiera ok. 69 MB. Nie wymaga konta ani GPU.
cache = Path('cache'); cache.mkdir(exist_ok=True)
embedding_path = cache/f'{MODEL_NAME}.txt.gz'
EMBEDDING_URL = f'https://github.com/piskvorky/gensim-data/releases/download/{MODEL_NAME}/{MODEL_NAME}.gz'
if not embedding_path.exists():
    print('Pobieranie GloVe...')
    urllib.request.urlretrieve(EMBEDDING_URL, embedding_path)
emb = Embeddings.load(str(embedding_path), verbose=True)
emb.normalize(l2=True, abtt=1)  # długość 1, usunięcie dominującego kierunku, ponowne L2
print('✓ Embeddings created:', emb.vectors.shape)

### ▶️ Krok kodu 10

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** komunikat lub wyniki tekstowe pod komórką.

In [ ]:
# Kontrola pokrycia: słowa poza słownikiem nie dostają sztucznego wektora.
all_context_tokens = [t for ts in contexts.tokens for t in ts]
covered = sum(t in emb for t in all_context_tokens)
coverage = covered / max(1,len(all_context_tokens))
print(f'Pokrycie tokenów kontekstu: {covered}/{len(all_context_tokens)} = {coverage:.1%}')
print('Przykładowe słowa poza słownikiem:', sorted({t for t in all_context_tokens if t not in emb})[:15])

Wektor słowa ma 50 współrzędnych bez prostych etykiet psychologicznych. Słowa podobne
dystrybucyjnie mają zbliżone kierunki. **Cosine similarity** mierzy zgodność kierunków.
L2 nadaje długość 1. **ABTT** usuwa dominujący, ogólny kierunek wspólny dla wielu słów.
To inny krok niż późniejsze PCA na wektorach rekordów.

GloVe 50d wybieramy dla szybkości i pamięci w Colab, nie jako najlepszy model SSD.
Model jest starszy i anglojęzyczny. MiniLM z modułu 04 reprezentował całe teksty.
Tutaj statyczna przestrzeń słów pozwala bezpośrednio wyszukiwać sąsiadów ±β zgodnie z SSD.

**✅ Co powinno wyjść?** 400 000 wektorów po 50 liczb dla domyślnego modelu i empiryczne pokrycie kontekstów.

**🔎 Jak to interpretować?** Embeddingi są uprzednio wyuczone na zewnętrznym korpusie. Nie uczymy słownika na wynikach osobowości.

**⚠️ Na co uważać?** Geometria zależy od modelu i normalizacji. Sąsiedztwo słów nie jest „prawdziwym znaczeniem”. MODEL_NAME obsługuje tekstowe dystrybucje GloVe z gensim-data. Inne formaty wymagają dostosowania pobierania.

## 🔎 7. Wektor pojęcia dla każdego rekordu

**▶️ Co teraz robimy?** Budujemy PCV autorskim pakietem i pokazujemy obliczenie średniej.

**🎯 Po co to robimy?** Jedna reprezentacja pojęcia przypada na jeden rekord z użytecznym kontekstem.

### ▶️ Krok kodu 11

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** komunikat lub wyniki tekstowe pod komórką.

In [ ]:
y_all = pd.to_numeric(df[BIG_FIVE_TRAIT], errors='coerce').to_numpy(dtype=float)
if not np.isfinite(y_all).all():
    raise SystemExit('Wybrany Y zawiera braki lub wartości nienumeryczne. Usuń te rekordy przed sekcją 4.')
ssd = SSD(emb, corpus, y=y_all, lexicon=[TARGET_CONCEPT],
          window=CONTEXT_WINDOW, sif_a=SIF_A)
X, y = ssd.x, ssd.y
kept_rows = np.flatnonzero(ssd._keep_mask)
ids = df.iloc[kept_rows].ID.to_numpy()
if len(y) < max(20, 4*PCA_MAX) or np.std(y) < 1e-12:
    raise SystemExit('Za mało rekordów względem K lub stała zmienna Y. Popraw ustawienia.')
print('✓ Concept vectors created:', X.shape)

**SIF:** częste słowa otrzymują mniejszą wagę: a / (a + częstość względna słowa).
Wagi wynikają z tokenów całego wybranego pytania, a=0,001. Dla każdego wystąpienia
uśredniamy wektory sąsiadów z tymi wagami. Potem uśredniamy konteksty jednego rekordu
i normalizujemy wynik do długości 1. Rekord bez użytecznych sąsiadów wypada.

### ▶️ Krok kodu 12

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
# Rozpisanie SIF dla zajęć. Wynik porównamy z macierzą autorskiego SSD.
word_counts, total_tokens = compute_global_sif(corpus.docs)
def context_vector(tokens, counts, total):
    known = [t for t in tokens if t in emb]
    if not known:
        return None
    weights = np.array([SIF_A/(SIF_A+counts.get(t,0)/max(1,total)) for t in known])
    return np.average(np.array([emb[t] for t in known]), axis=0, weights=weights)
contexts['vector'] = [context_vector(ts,word_counts,total_tokens) for ts in contexts.tokens]
contexts = contexts[contexts.vector.notna()].copy()
manual = np.vstack([np.mean(np.stack(contexts.loc[contexts.ID==i,'vector']),axis=0) for i in ids])
manual /= np.linalg.norm(manual,axis=1,keepdims=True)
assert np.allclose(X,manual,atol=1e-8)
context_counts = contexts.groupby('ID').size().reindex(ids)
vectors = pd.DataFrame(X, columns=[f'dim_{j:02d}' for j in range(X.shape[1])])
vectors.insert(0,'ID',ids); vectors['n_contexts']=context_counts.to_numpy()
vectors.to_csv(OUT/'participant_concept_vectors.csv',index=False)
contexts.drop(columns=['tokens','vector']).to_csv(OUT/'contexts.csv',index=False)
np.save(OUT/'context_vectors.npy',np.stack(contexts.vector))
display(context_counts.value_counts().sort_index().rename('n_records').to_frame())

**✅ Co powinno wyjść?** Macierz PCV i rozkład liczby kontekstów. Ręczne obliczenie zgadza się z pakietem.

**🔎 Jak to interpretować?** Średnia współrzędna po współrzędnej zbiera lokalne użycia pojęcia w jednym rekordzie. ID jest jednostką pliku, nie zweryfikowaną osobą.

**⚠️ Na co uważać?** Nazwy PCV/participant w metodzie i eksporcie nie potwierdzają autentyczności uczestników. Średnia może ukryć sprzeczne użycia. Pliki eksportują tylko rekordy rzeczywiście użyte.

## 🔎 8. Wybór wyniku cechy

**▶️ Co teraz robimy?** Oglądamy rozkład Y w analizowanej podpróbie.

**🎯 Po co to robimy?** Próba z pojęciem może różnić się od całego zbioru.

### ▶️ Krok kodu 13

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami oraz wykres.

In [ ]:
outcome_counts = pd.Series(y).value_counts().sort_index()
display(outcome_counts.rename('n').to_frame())
print('N użyte:',len(y),'z',len(df))
plt.figure(); plt.bar(outcome_counts.index,outcome_counts.values,color='#247F87')
plt.xticks(range(1,6)); plt.xlabel(BIG_FIVE_TRAIT); plt.ylabel('Liczba rekordów')
plt.tight_layout(); plt.savefig(OUT/'trait_distribution.png',dpi=160); plt.show()

**✅ Co powinno wyjść?** Rozkład ocen 1–5 i N z pojęciem.

**🔎 Jak to interpretować?** Conscientiousness jest tu etykietą w pliku. OLS przyjmuje równe odległości między poziomami, chociaż pomiar jest porządkowy. Pakiet standaryzuje Y.

**⚠️ Na co uważać?** Nie medianizujemy Y. Wyniki pochodzą z ocen tekstu, a nie udokumentowanego niezależnego kwestionariusza. Inne pytanie dla Y nie usuwa kolistości.

## 🔎 9. Kierunek związany z wynikiem cechy

**▶️ Co teraz robimy?** Dopasowujemy PCA + OLS funkcją autora.

**🎯 Po co to robimy?** PCA ogranicza liczbę predyktorów, a OLS szacuje związek reprezentacji z Y.

Najpierw pakiet standaryzuje współrzędne PCV i Y. PCA zachowuje K głównych
kierunków zróżnicowania PCV. OLS przewiduje standaryzowane Y z wybranych składowych.
Współczynniki wracają z przestrzeni PCA do przestrzeni embeddingów i uwzględniają
wcześniejsze skalowanie. Powstaje **β**, a po podzieleniu przez długość: **gradient**.

**PCA sweep z wersji v3:** porównujemy K=1,3,…,15. Pakiet ocenia jakość klastrów
słów przy biegunach, uwzględnia pojemność reprezentacji i stabilność kierunku między
sąsiednimi K. Wygładza wyniki, a następnie wybiera najmniejsze K z najlepszym łącznym
kryterium. Nie wybiera K według najmniejszego p ani największego R².
Zakres 1–15 jest ograniczeniem dydaktycznym dla 50 wymiarów i tej próby. W badaniach
autorów użyto polskiego modelu 800d i szerszego zakresu. Nie replikujemy ich korpusów.

### ▶️ Krok kodu 14

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
result = ssd.fit_ols(fixed_k=PCA_K,k_min=1,k_max=PCA_MAX,k_step=2)
SELECTED_K = result.pca_k
result.sweep.to_df().to_csv(OUT/'pca_sweep.csv',index=False)
display(result.sweep.to_df())
beta, direction = result.beta, result.gradient
display(result.stats.to_df())
result.stats.to_df().to_csv(OUT/'model_stats.csv',index=False)
pd.DataFrame({'dimension': range(len(beta)), 'beta': beta, 'unit_direction': direction,
              'pcv_mean': X.mean(axis=0)}).to_csv(OUT/'semantic_direction.csv',index=False)
print('✓ Regression fitted — PCA + OLS, wybrane K =',SELECTED_K)
print('✓ Semantic gradient estimated')

**✅ Co powinno wyjść?** Statystyki autorskiego modelu i współczynniki w przestrzeni słów.

**🔎 Jak to interpretować?** R² dopasowania opisuje tę próbę. P-value testu F pochodzi z OLS i jego założeń; nie stanowi testu trafności psychometrycznej.

**⚠️ Na co uważać?** Przy małym N wiele parametrów grozi przeuczeniem. Zakres K ograniczamy z góry. Test F po wyborze K ma charakter nominalny, nie uwzględnia całej selekcji modelu. Nie ukrywamy słabego dopasowania ani nie zmieniamy Y na podstawie wyniku.

## 🔎 10. Położenie rekordów na kierunku

**▶️ Co teraz robimy?** Umieszczamy rekordy i słowa wzdłuż gradientu.

**🎯 Po co to robimy?** Oś porządkuje reprezentacje względem oszacowanego związku z Y.

### ▶️ Krok kodu 15

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
# PCV i gradient mają długość 1: iloczyn skalarny jest cosine alignment.
alignment = X @ direction
fitted_y = y.mean() + y.std(ddof=0) * ((X-X.mean(axis=0)) @ beta)
assert np.isclose(r2_score(y,fitted_y),result.stats.r2)
results = pd.DataFrame({'ID':ids,'outcome':y,'alignment':alignment,'fitted_y':fitted_y,
                        'n_contexts':context_counts.to_numpy()})
results.to_csv(OUT/'semantic_gradient_results.csv',index=False)
display(results.head())
words = result.words.to_df()
words.to_csv(OUT/'pole_words.csv',index=False)
display(pd.concat([words[words.side==s].head(8) for s in ['neg','pos']]))

**✅ Co powinno wyjść?** Położenie rekordów na osi i słowa przy ±β.

**🔎 Jak to interpretować?** Wyższy alignment oznacza wyższą predykcję końcowego modelu, nie koniecznie wyższy rzeczywisty Y. Słowa sąsiadujące pochodzą ze słownika GloVe.

**⚠️ Na co uważać?** Te słowa nie muszą występować w badanym korpusie. Nie wolno przedstawiać ich jako słów uczestników. Oś jest nadzorowana przez Y, więc związek na treningu nie jest walidacją.

## 🔎 11. Czytanie kontekstów przy biegunach

**▶️ Co teraz robimy?** Łączymy bieguny, klastry słów i oryginalne wypowiedzi.

**🎯 Po co to robimy?** To część interpretacyjna SSD, konieczna do nazwania wzorca semantycznego.

### ▶️ Krok kodu 16

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
# Klasteryzacja sąsiadów osobno na każdym biegunie, funkcją pakietu autora.
pos_clusters = result.clusters.pos(topn=100,k_min=2,k_max=5)
neg_clusters = result.clusters.neg(topn=100,k_min=2,k_max=5)
clusters = pd.concat([neg_clusters.to_df().assign(side='neg'),pos_clusters.to_df().assign(side='pos')],ignore_index=True)
clusters.to_csv(OUT/'pole_clusters.csv',index=False)
display(clusters)
cluster_words = pd.concat([neg_clusters.words.to_df().assign(side='neg'),pos_clusters.words.to_df().assign(side='pos')],ignore_index=True)
cluster_words.to_csv(OUT/'cluster_words.csv',index=False)

### ▶️ Krok kodu 17

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
# Dosłowne wypowiedzi i dokładne lemma-konteksty z naszego obliczenia PCV.
cvectors = np.stack(contexts.vector)
cvectors = cvectors/np.linalg.norm(cvectors,axis=1,keepdims=True)
projected = contexts.drop(columns=['tokens','vector']).assign(alignment=cvectors@direction)
projected = projected.merge(results[['ID','outcome']],on='ID',validate='many_to_one')
low = projected.sort_values('alignment').drop_duplicates('context_tokens').drop_duplicates('ID').head(5).assign(pole='lower')
high = projected.sort_values('alignment',ascending=False).drop_duplicates('context_tokens').drop_duplicates('ID').head(5).assign(pole='higher')
extremes = pd.concat([low,high],ignore_index=True)
extremes.to_csv(OUT/'extreme_contexts.csv',index=False)
display(extremes[['pole','ID','outcome','alignment','context_tokens','full_text']])

### ▶️ Krok kodu 18

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
# Dodatkowe powiązanie klastrów z tekstem przez autorską funkcję snippetów.
snippets = pd.concat([neg_clusters.snippets.to_df(cols='all').assign(side='neg'),pos_clusters.snippets.to_df(cols='all').assign(side='pos')],ignore_index=True)
snippets['ID'] = df.iloc[snippets.doc_id.astype(int)].ID.to_numpy()
snippets.to_csv(OUT/'cluster_snippets.csv',index=False)
display(snippets.head(6))

**✅ Co powinno wyjść?** Sąsiedzi osi, ich klastry, dziesięć skrajnych kontekstów i teksty przy klastrach.

**🔎 Jak to interpretować?** Najpierw opisz język obu końców, potem zaproponuj ostrożne nazwy. Porównaj lemma-kontekst z pełną wypowiedzią i sprawdź, czy klastry pomagają w interpretacji.

**⚠️ Na co uważać?** Klastry sąsiadów SSD nie są tematami dokumentów BERTopic. Skrajne teksty wybrano na tej samej próbie. Jeśli słowa zewnętrznego słownika są nieadekwatne, odnotuj to zamiast wymuszać interpretację.

## 🔎 12. Sprawdzenie wpływu ustawień

**▶️ Co teraz robimy?** Sprawdzamy przewidywanie poza treningiem oraz wrażliwość na okno i K.

**🎯 Po co to robimy?** Oddzielamy atrakcyjną interpretację treningową od zdolności uogólnienia.

**Walidacja 5-fold:** rekordy z identycznym zestawem lemma-kontekstów okna głównego
pozostają w jednej grupie. W każdym foldzie częstości SIF, standaryzację, PCA i OLS
wyznaczamy wyłącznie na treningu. Model embeddingów i jego normalizacja są zewnętrzne
i stałe. Również wybór K przez sweep wykonujemy osobno na treningu. Baseline przewiduje średnią Y z treningu.
To dodatkowa kontrola warsztatowa, nie nazwa nowego algorytmu SSD.

### ▶️ Krok kodu 19

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** przygotowanie funkcji lub ustawień do dalszych kroków; brak wydruku jest prawidłowy.

In [ ]:
# Testowe wektory muszą używać częstości SIF z treningu.
def transform_test(test_docs, train_docs, window):
    wc, total = compute_global_sif(train_docs)
    xx, keep = build_doc_vectors(test_docs, emb, {TARGET_CONCEPT}, wc, total, window, SIF_A)
    if not keep.all():
        raise ValueError('Brak wektora testowego. Sprawdź pokrycie słownika.')
    return xx/np.maximum(np.linalg.norm(xx,axis=1,keepdims=True),1e-12)
selected_docs = [corpus.docs[i] for i in kept_rows]
groups = contexts.groupby('ID').context_tokens.apply(lambda s: ' || '.join(sorted(s))).reindex(ids).to_numpy()
if np.unique(groups).size < 10:
    raise SystemExit('Za mało różnych odpowiedzi do walidacji.')

### ▶️ Krok kodu 20

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
cv_selections = {}
def cross_validate(window, k):
    pred, baseline, folds = np.zeros(len(y)), np.zeros(len(y)), np.zeros(len(y),dtype=int)
    chosen = []
    for fold,(train,test) in enumerate(GroupKFold(5).split(np.zeros(len(y)),y,groups),1):
        train_docs = [selected_docs[i] for i in train]
        train_corpus = Corpus(train_docs,pretokenized=True,lang='en')
        fit_ssd = SSD(emb,train_corpus,y[train],lexicon=[TARGET_CONCEPT],window=window,sif_a=SIF_A)
        fit_result = fit_ssd.fit_ols(fixed_k=k,k_min=1,k_max=PCA_MAX,k_step=2)
        chosen.append(fit_result.pca_k)
        xtest = transform_test([selected_docs[i] for i in test],train_docs,window)
        pred[test] = y[train].mean()+y[train].std(ddof=0)*((xtest-fit_ssd.x.mean(axis=0))@fit_result.beta)
        baseline[test] = y[train].mean()
        folds[test] = fold
    cv_selections[f'{window}:{k}'] = chosen
    return pred,baseline,folds
oof,baseline,folds = cross_validate(CONTEXT_WINDOW,PCA_K)
results['oof_y']=oof; results['baseline_y']=baseline; results['fold']=folds
results.to_csv(OUT/'semantic_gradient_results.csv',index=False)
metrics = {'cv_r2':r2_score(y,oof),'cv_mae':mean_absolute_error(y,oof),
           'baseline_r2':r2_score(y,baseline),'baseline_mae':mean_absolute_error(y,baseline)}
display(pd.Series(metrics).to_frame('value'))

### ▶️ Krok kodu 21

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
alternative_ssd = SSD(emb,corpus,y_all,lexicon=[TARGET_CONCEPT],window=ALTERNATIVE_WINDOW,sif_a=SIF_A)
if not np.array_equal(ssd._keep_mask,alternative_ssd._keep_mask):
    raise SystemExit('Okna zachowały inne rekordy. Porównaj modele na ich części wspólnej.')
alternative_result = alternative_ssd.fit_ols(fixed_k=PCA_K,k_min=1,k_max=PCA_MAX,k_step=2)
alt_oof,alt_base,alt_folds = cross_validate(ALTERNATIVE_WINDOW,PCA_K)
alt_alignment = alternative_ssd.x@alternative_result.gradient
stability = {'gradient_cosine':float(direction@alternative_result.gradient),
             'alignment_spearman':float(spearmanr(alignment,alt_alignment).statistic)}
sensitivity = pd.DataFrame([
    {'window':CONTEXT_WINDOW,'k':SELECTED_K,'n':len(y),'train_r2':result.stats.r2,**metrics},
    {'window':ALTERNATIVE_WINDOW,'k':alternative_result.pca_k,'n':len(y),'train_r2':alternative_result.stats.r2,
     'cv_r2':r2_score(y,alt_oof),'cv_mae':mean_absolute_error(y,alt_oof),
     'baseline_r2':r2_score(y,alt_base),'baseline_mae':mean_absolute_error(y,alt_base)}])
sensitivity.to_csv(OUT/'window_sensitivity.csv',index=False)
pd.DataFrame({'ID':ids,'alignment':alt_alignment,'oof_y':alt_oof,'fold':alt_folds}).to_csv(OUT/'alternative_results.csv',index=False)
display(sensitivity); print(stability)

### ▶️ Krok kodu 22

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
# Prosta kontrola decyzji K. Nie zmienia modelu głównego.
k_rows=[]
for k in sorted(set([3,SELECTED_K,7])):
    check = ssd.fit_ols(fixed_k=k)
    pred_k,_,_ = cross_validate(CONTEXT_WINDOW,k)
    k_rows.append({'k':k,'train_r2':check.stats.r2,'cv_r2':r2_score(y,pred_k),
                  'gradient_cosine_to_main':float(check.gradient@direction)})
k_sensitivity = pd.DataFrame(k_rows)
k_sensitivity.to_csv(OUT/'k_sensitivity.csv',index=False)
display(k_sensitivity)

### ▶️ Krok kodu 23

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** wykres.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].scatter(y,oof,color='#247F87',alpha=.6)
axes[0].plot([1,5],[1,5],'--',color='gray')
axes[0].set(xlabel='Obserwowana ocena Y',ylabel='Predykcja poza treningiem')
axes[1].bar(sensitivity.window.astype(str),sensitivity.cv_r2,color=['#247F87','#C58B39'])
axes[1].axhline(0,color='gray',lw=1)
axes[1].set(xlabel='Okno po każdej stronie',ylabel='R² poza treningiem')
fig.tight_layout(); fig.savefig(OUT/'validation.png',dpi=160); plt.show()

**✅ Co powinno wyjść?** R² i MAE poza treningiem, baseline, dwa okna oraz kontrolne K=3 i K=7.

**🔎 Jak to interpretować?** R² ujemne oznacza gorszą predykcję niż średnia całej ocenianej próby. MAE ma jednostkę punktów oceny. Cosine kierunków i Spearman kolejności opisują stabilność.

**⚠️ Na co uważać?** Nie wybieramy po fakcie najładniejszego K lub okna. Duplikaty grupujemy, lecz podobne szablony mogą nadal przeciekać. Zależne etykiety nie stają się niezależne przez cross-validation.

## 🔎 13. Interpretacja

**▶️ Co teraz robimy?** Zapisujemy konfigurację i wnioski.

**🎯 Po co to robimy?** Każda liczba i cytat powinny mieć ścieżkę do wykonanej analizy.

### ▶️ Krok kodu 24

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** komunikat lub wyniki tekstowe pod komórką.

In [ ]:
summary = {'dataset':DATA_PATH.name,'dataset_n':len(df),'columns':list(df.columns),
           'missingness':df.isna().sum().to_dict(),'target_concept':TARGET_CONCEPT,
           'text_column':TEXT_COLUMN,'outcome':BIG_FIVE_TRAIT,'n_used':len(y),
           'n_contexts':len(contexts),'context_count_distribution':context_counts.value_counts().sort_index().to_dict(),
           'outcome_counts':outcome_counts.to_dict(),'model':MODEL_NAME,'embedding_dimensions':X.shape[1],
           'context_coverage':coverage,'window':CONTEXT_WINDOW,'alternative_window':ALTERNATIVE_WINDOW,
           'pca_k':SELECTED_K,'pca_range':[1,PCA_MAX,2],'cv_selected_k':cv_selections,'sif_a':SIF_A,'cv_groups':int(np.unique(groups).size),
           'train_r2':float(result.stats.r2),'train_p':float(result.stats.pvalue),
           'train_r2_adj':float(result.stats.r2_adj),**metrics,**stability,
           'python':platform.python_version(),
           'versions':{p:meta.version(p) for p in ['ssdiff','spacy','en-core-web-sm','numpy','pandas','scipy','scikit-learn','matplotlib']},
           'dataset_sha256':hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
           'embedding_sha256':hashlib.sha256(embedding_path.read_bytes()).hexdigest()}
(OUT/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print('✓ Results saved in',OUT)
print(f'N={len(y)}, R² poza treningiem={metrics["cv_r2"]:.3f}, MAE={metrics["cv_mae"]:.3f}')

**Schemat wypowiedzi:** „W analizowanych rekordach konteksty pojęcia układają się w
oszacowany kierunek związany z przypisaną oceną. Po przeczytaniu tekstów opisuję
bieguny jako … . Wynik walidacji … oraz porównanie okien … ograniczają uogólnienie”.

Nie twierdzimy, że odzyskaliśmy prawdziwe znaczenie lub zmierzyliśmy osobowość.
Wynik zależy od embeddingów, pojęcia, okna, preprocessingu, agregacji, Y, próby
i regresji. Dane nie dokumentują niezależnego pomiaru cechy, przyczynowości ani diagnozy.

**✅ Co powinno wyjść?** summary.json i pliki wynikowe.

**🔎 Jak to interpretować?** SSD łączy ilościowy model kierunku z jakościową interpretacją jego otoczenia.

**⚠️ Na co uważać?** Interpretacja z tej samej próby jest eksploracyjna. Zmiana parametrów zastępuje pliki wynikowe; dla ćwiczenia użyj kopii notebooka.

**Komentarz do zapisanego wykonania domyślnego (stress, q5, Conscientiousness):**
PCA sweep wybrał K=11 w ustalonym zakresie. Walidacja całej procedury daje R² około
0,214 i MAE 0,829 punktu, wobec 0,915 dla średniej treningowej. Przy ujemnym końcu
osi teksty mówią o rutynie i znajomych zadaniach (np. ID 310), przy dodatnim o
odzyskiwaniu opanowania, skupieniu i spokoju (np. ID 35, 224). Nie każdy przypadek
pasuje do tego opisu: ID 8 wspomina też przeciążenie. Część zewnętrznych klastrów
GloVe obejmuje słowa nieadekwatne do stresu, np. flip/hooks lub nazwy miejsc.
To ogranicza interpretowalność osi pomimo lepszej od baseline predykcji.
Zależne od tekstu pochodzenie ocen uniemożliwia traktowanie wyniku jako niezależnej
walidacji sumienności. Po zmianie parametrów ten komentarz nie opisuje nowej analizy.

## 🔎 14. Ćwiczenie

**▶️ Co teraz robimy?** Zmieniamy pojęcie i cechę oraz ponawiamy analizę.

**🎯 Po co to robimy?** Ćwiczymy decyzje metodologiczne i czytanie biegunów.

1. W tabeli częstości znajdź inny koncept, np. `team` w `q3_response`.
2. Zmień `TARGET_CONCEPT`, `TEXT_COLUMN` i `BIG_FIVE_TRAIT`, np. na `Agreeableness_Score`.
3. Uruchom wszystkie komórki od sekcji 4 do końca.
4. Porównaj słowa, klastry i wypowiedzi przy obu biegunach oraz wyniki walidacji.
5. Napisz interpretację w 2–3 zdaniach.
6. Wskaż alternatywne wyjaśnienie lub ograniczenie, np. punktację wyprowadzoną z tekstu.

**Twoja odpowiedź:** …

**✅ Co powinno wyjść?** Krótka interpretacja nowego konceptu i przynajmniej jedno ograniczenie.

**🔎 Jak to interpretować?** SSD pozostaje ostatnią metodą warsztatu. Wspólną zasadą jest powrót od modelu do tekstu.

**⚠️ Na co uważać?** Eksploracji nie przedstawiaj jako potwierdzenia hipotezy. Nie wybieraj ustawień wyłącznie dla małego p.

### 🔎 Źródła
- Hubert Plisiecki, *AI in the study (1).pptx*, slajdy 14–15 i 20–24, dostarczony materiał.
- Plisiecki, Lenartowicz, Pokropek, Małyska i Flakus, *Measuring Individual Differences in Meaning: The Supervised Semantic Differential*, [preprint wskazany przez autora](https://doi.org/10.31234/osf.io/gvrsb_v3).
- [Kod i dokumentacja autora](https://github.com/hplisiecki/Supervised-Semantic-Differential), `ssdiff 3.0.0`, funkcje `SSD.fit_ols`, budowa wektorów SIF i interpretacja wyników.
- Plisiecki, Leniarska, Piotrowski i Zajenkowski (2026), [Interpretable Semantic Gradients in SSD](https://arxiv.org/abs/2603.13038), kontekst PCA i wyboru liczby składowych.
- Pennington, Socher i Manning (2014), [GloVe](https://nlp.stanford.edu/projects/glove/). Wektory Wikipedia 2014 + Gigaword, wersja 50d. [Dystrybucja pliku](https://github.com/piskvorky/gensim-data).
- [Kaggle, Personality Trait & Categories Based on Big Five](https://www.kaggle.com/datasets/hassan1212/personality-trait-and-categories-based-on-big-five).

## 💾 Pobranie wyników

Uruchom tę komórkę po ukończeniu analizy. Utworzy ZIP i w Colab rozpocznie pobieranie. Jeśli przeglądarka je zablokuje, odszukaj ZIP w panelu Pliki i pobierz ręcznie. Zachowaj też własną kopię notebooka.

In [ ]:
import shutil
bundle = Path('wyniki_modul_05')
bundle.mkdir(exist_ok=True)
if not Path(OUT).exists():
    raise RuntimeError('⛔ Najpierw wykonaj komórki analizy i zapisu wyników.')
shutil.copytree(OUT, bundle / 'tabele_i_wykresy', dirs_exist_ok=True)
archive = shutil.make_archive(str(bundle), 'zip', bundle)
print('✅ Plik wyników:', archive)
if IN_COLAB:
    from google.colab import files
    files.download(archive)
